# Notebook 08 — Phase 2: Pipeline Evaluation


## Objective

This notebook answers the Phase 2 questions related to the *end-to-end retrieval pipeline*:
1. How can the classifier be used to improve retrieval?
2. How does the full pipeline (retrieval + classification + reranking) perform?
3. Does reranking actually improve the retrieval metrics?
4. Which retrieval configuration (TF‑IDF, BM25+, Embeddings, Hybrid) works best in this setup?

## Structure

- **Section 1**: Classifier training  
- **Section 2**: Retrieval baselines on train queries (TF‑IDF, BM25+, Embeddings)  
- **Section 3**: Retrieval + classification-based reranking (hard_filter, soft_boost, Hybrid RRF)  
- **Section 4**: Comparison with/without reranking and discussion  
- **Section 5**: Sensitivity analysis of soft_boost (effect of the boost_factor)
- **Section 6**: Additional experiments
- **Section 7**: Cross-Encoder reranking (neural reranker on top-K candidates)



## 1. Classifier Training

The full classifier ablation (NB, SVC, LogReg, MLP × TF-IDF / Count / Embeddings) is conducted in **Notebook 07**.  
Conclusion from the ablation: **TF-IDF + LinearSVC** achieves the best macro-F1 (0.9852) but lacks `predict_proba`,  
which is required for the **soft_boost** reranking strategy.

We therefore use **TF-IDF + Logistic Regression** in this notebook:
- macro-F1 = 0.9819 on val set (gap of ~0.003 vs SVC, negligible)
- Provides calibrated `predict_proba` → enables both hard_filter and soft_boost
- Good training speed (~15s)


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
project_root = cwd.parent if (cwd.parent / 'src').exists() else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.kaggle.submit_phase2 import set_global_seeds
from src.config import RANDOM_SEED
set_global_seeds(RANDOM_SEED)
print(f'Seeds fixed: {RANDOM_SEED}')

In [ ]:
from src.data.load import load_all
from src.data.preprocess import add_content_field
from src.evaluation.evaluate import adapt_ground_truth, adapt_ground_truth_categories

raw_dir = project_root / 'data' / 'raw'
docs, queries_train, queries_test, gts_raw = load_all(raw_dir)
docs, queries_train = add_content_field(docs, queries_train,clean=True)

gt = adapt_ground_truth(gts_raw)
gt_categories = adapt_ground_truth_categories(gts_raw)

query_ids_train = [q['id'] for q in queries_train]
print(f'Docs: {len(docs)}, Train queries: {len(queries_train)}')

### Data and Split

In [ ]:
from src.classification.features import extract_texts_and_labels, build_tfidf_features, stratified_split
from src.classification.interfaces import CATEGORIES
from src.config import RANDOM_SEED

texts, labels = extract_texts_and_labels(docs, queries_train, text_field="content")
print(f"Total samples : {len(texts)}")
for cat in CATEGORIES:
    print(f"  {cat:15s}: {labels.count(cat)}")

X_train, y_train, X_val, y_val, X_test, y_test = stratified_split(
    texts, labels, val_size=0.15, test_size=0.15, random_state=RANDOM_SEED
)
print(f"\nTrain : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")

clf_vectorizer, X_train_mat, X_val_mat, X_test_mat = build_tfidf_features(
    X_train, X_val, X_test,
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
)
print(f"Feature matrix (train) : {X_train_mat.shape}")

### LogReg training and evaluation

In [ ]:
from src.classification.model import Classifier
from src.evaluation.metrics import accuracy, macro_f1, balanced_accuracy, classification_report_phase2, confusion_matrix_phase2
import time

clf = Classifier(method="logreg")
t0 = time.time()
clf.fit(X_train_mat, y_train)
print(f"LogReg trained in {time.time()-t0:.1f}s")

# Validation
y_val_pred = clf.predict(X_val_mat)
print(f"\n--- Validation set ---")
print(f"  Accuracy        : {accuracy(y_val, y_val_pred):.4f}")
print(f"  Macro-F1        : {macro_f1(y_val, y_val_pred):.4f}")
print(f"  Balanced acc    : {balanced_accuracy(y_val, y_val_pred):.4f}")

# Test
y_test_pred = clf.predict(X_test_mat)
print(f"\n--- Test set ---")
print(classification_report_phase2(y_test, y_test_pred))


In [ ]:
from src.evaluation.evaluate import evaluate_docs_vs_queries

doc_texts   = [d["content"] for d in docs]
doc_labels  = [d["category"] for d in docs]
q_texts     = [q["content"] for q in queries_train]
q_labels    = [q["category"] for q in queries_train]

X_docs_mat    = clf_vectorizer.transform(doc_texts)
X_queries_mat = clf_vectorizer.transform(q_texts)

dv = evaluate_docs_vs_queries(clf, X_docs_mat, doc_labels, X_queries_mat, q_labels)
for split, m in dv.items():
    print(f"\n[{split}]")
    print(f"  accuracy     : {m['accuracy']:.4f}")
    print(f"  macro_f1     : {m['macro_f1']:.4f}")
    print(f"  balanced_acc : {m['balanced_accuracy']:.4f}")


## 2. Baseline retrieval (without reranking)

In this section, we reproduce the **pure retrieval baselines** from Phase 1:

- TF-IDF retrieval
- BM25+ retrieval
- Embeddings-based retrieval

For each method, we evaluate on the **train queries** with all the documents  
and compute standard IR metrics at cut-off `K` (here: `K = 100`):

- Recall@K
- Precision@K
- MRR@K

These scores will serve as **reference baselines** to compare against the classifier-based reranking in Section 3.


In [ ]:
MAX_DOCS = 216_041

docs_subset = docs[:MAX_DOCS]
print(f"Subsampled docs: {len(docs_subset)} / {len(docs)} total")


In [ ]:
from pathlib import Path
import time

from src.retrieval.tfidf import fit_tfidf, retrieve_tfidf, map_indices_to_docids as map_tfidf
from src.retrieval.bm25 import fit_bm25, retrieve_bm25, map_indices_to_docids as map_bm25
from src.retrieval.embeddings import build_embeddings, retrieve_embeddings
from src.evaluation.evaluate import evaluate_run

K = 100
TEXT_FIELD = "content"
MODEL_NAME = "all-MiniLM-L12-v2"
CACHE_DIR = project_root / "data" / "cache"

print(f"Number of docs (subset) : {len(docs_subset)}")
print(f"Number of train queries  : {len(queries_train)}")

# ===================== TF-IDF =====================
t0 = time.time()
tfidf_vectorizer, tfidf_doc_matrix = fit_tfidf(docs_subset, text_field=TEXT_FIELD)
print(f"[TF-IDF] fit done in {time.time()-t0:.2f}s")

t0 = time.time()
topk_indices_tfidf, topk_scores_tfidf = retrieve_tfidf(
    tfidf_vectorizer, tfidf_doc_matrix, queries_train, k=K, text_field=TEXT_FIELD
)
print(f"[TF-IDF] retrieve done in {time.time()-t0:.2f}s")

pred_docids_tfidf = map_tfidf(topk_indices_tfidf, docs_subset)
metrics_tfidf = evaluate_run(pred_docids_tfidf, gt, query_ids_train, K)

# ===================== BM25 Plus & Okapi =====================
t0 = time.time()
bm25plus_model = fit_bm25(docs_subset, text_field=TEXT_FIELD, method="plus")
bm25okapi_model = fit_bm25(docs_subset, text_field=TEXT_FIELD, method="okapi")
print(f"[BM25] fit (plus+okapi) done in {time.time()-t0:.2f}s")

t0 = time.time()
topk_indices_bm25p, topk_scores_bm25p = retrieve_bm25(
    bm25plus_model, docs_subset, queries_train, k=K, text_field=TEXT_FIELD
)
topk_indices_bm25o, topk_scores_bm25o = retrieve_bm25(
    bm25okapi_model, docs_subset, queries_train, k=K, text_field=TEXT_FIELD
)
print(f"[BM25] retrieve (plus+okapi) done in {time.time()-t0:.2f}s")

pred_docids_bm25p = map_bm25(topk_indices_bm25p, docs_subset)
pred_docids_bm25o = map_bm25(topk_indices_bm25o, docs_subset)
metrics_bm25p = evaluate_run(pred_docids_bm25p, gt, query_ids_train, K)
metrics_bm25o = evaluate_run(pred_docids_bm25o, gt, query_ids_train, K)

# ===================== Embeddings =====================
t0 = time.time()
doc_emb, query_emb = build_embeddings(
    docs_subset,
    queries_train,
    text_field=TEXT_FIELD,
    model_name=MODEL_NAME,
    batch_size=64,
    show_progress_bar=True,
    cache_dir=CACHE_DIR,
    device=None,              # "auto"
    precision="float32",
    truncate_dim=None,
    chunk_size=None,
    normalize_embeddings=True,
    backend="torch",
    model_max_seq_length=None,
    local_files_only=False,
    use_cache=True,
)
print(f"[Embeddings] build done in {time.time()-t0:.2f}s  shapes: docs={doc_emb.shape}, queries={query_emb.shape}")

t0 = time.time()
topk_indices_emb, topk_scores_emb = retrieve_embeddings(
    doc_emb,
    query_emb,
    k=K,
    assume_normalized=True,
)
print(f"[Embeddings] retrieve done in {time.time()-t0:.2f}s")

# For embeddings, we can reuse indices_to_docids directly.
from src.evaluation.evaluate import indices_to_docids
pred_docids_emb = indices_to_docids(topk_indices_emb, docs_subset)
metrics_emb = evaluate_run(pred_docids_emb, gt, query_ids_train, K)


In [ ]:
import pandas as pd

baseline_results = {
    "tfidf": {
        "precision@k": metrics_tfidf["precision@k"],
        "recall@k":    metrics_tfidf["recall@k"],
        "mrr@k":       metrics_tfidf["mrr@k"],
    },
    "bm25_plus": {
        "precision@k": metrics_bm25p["precision@k"],
        "recall@k":    metrics_bm25p["recall@k"],
        "mrr@k":       metrics_bm25p["mrr@k"],
    },
    "bm25_okapi": {
        "precision@k": metrics_bm25o["precision@k"],
        "recall@k":    metrics_bm25o["recall@k"],
        "mrr@k":       metrics_bm25o["mrr@k"],
    },
    "embeddings": {
        "precision@k": metrics_emb["precision@k"],
        "recall@k":    metrics_emb["recall@k"],
        "mrr@k":       metrics_emb["mrr@k"],
    },
}

df_baselines = pd.DataFrame(baseline_results).T
df_baselines.index.name = "method"
display(df_baselines.style.format(precision=4))


## 3. Retrieval + classification-based reranking

In this section, we evaluate how the classifier can improve retrieval quality.

We start from the four retrieval baselines computed in Section 2:
- TF-IDF
- BM25+ (plus)
- Embeddings (all-MiniLM-L12-v2)
- Hybrid RRF (BM25+ + Embeddings, weights: 0.5 / 3.5, RRF_k = 20)

For each retrieval method, we apply two classification-based reranking strategies:
- `hard_filter`: keep only documents whose category matches the predicted query category
- `soft_boost`: upweight scores of documents with high probability for the predicted category

We then compare, for each method, the retrieval metrics (precision@100, recall@100, MRR@100)
before and after reranking, as well as the query category accuracy.


In [ ]:
from src.classification.rerank import hard_filter, soft_boost, build_docs_index
from src.evaluation.evaluate import evaluate_with_classification, indices_to_docids
from src.retrieval.hybrid import fuse_rankings_rrf
from src.evaluation.evaluate import evaluate_with_classification
import pandas as pd

HYBRID_WEIGHT_EMBEDDINGS = 3.0
HYBRID_WEIGHT_BM25       = 1.0
HYBRID_RRF_K             = 20

K = 100

docs_by_id = build_docs_index(docs_subset)

hybrid_indices, hybrid_scores = fuse_rankings_rrf(
    emb_indices       = topk_indices_emb,
    bm25_indices      = topk_indices_bm25p,
    k_out             = K,
    rrf_k             = HYBRID_RRF_K,
    weight_embeddings = HYBRID_WEIGHT_EMBEDDINGS,
    weight_bm25       = HYBRID_WEIGHT_BM25,
)

# Map indices -> doc_ids for each retrieval method
pred_docids_tfidf   = indices_to_docids(topk_indices_tfidf,   docs_subset)
pred_docids_bm25p   = indices_to_docids(topk_indices_bm25p,   docs_subset)
pred_docids_emb     = indices_to_docids(topk_indices_emb,     docs_subset)
pred_docids_hybrid = indices_to_docids(hybrid_indices, docs_subset)

# Precompute predicted categories + probabilities for all train queries
predicted_cats = {}
predicted_probas = {}

for q in queries_train:
    qid = q["id"]
    q_vec = clf_vectorizer.transform([q["content"]])
    pred_label = clf.predict(q_vec)[0]
    proba = clf.predict_proba(q_vec)[0]   # shape (n_classes,)
    predicted_cats[qid] = pred_label
    predicted_probas[qid] = proba


In [ ]:
from src.classification.rerank import hard_filter, soft_boost, build_docs_index
from src.evaluation.evaluate import evaluate_run, evaluate_with_classification

docs_by_id = build_docs_index(docs_subset)

results_rows = []

# Group all retrieval methods, including the hybrid run.
retrieval_runs = [
    ("tfidf",      pred_docids_tfidf,  topk_scores_tfidf),
    ("bm25_plus",  pred_docids_bm25p,  topk_scores_bm25p),
    ("embeddings", pred_docids_emb,    topk_scores_emb),
    ("hybrid_rrf", pred_docids_hybrid, hybrid_scores),
]

for method_name, base_docids, base_scores in retrieval_runs:
    # 1) No reranking: evaluate_with_classification (Phase 2).
    metrics_base = evaluate_with_classification(
        base_docids,
        gt,
        query_ids_train,
        K,
        predicted_categories=predicted_cats,
        gt_categories=gt_categories,
    )
    results_rows.append({
        "method": method_name,
        "rerank": "no_rerank",
        **metrics_base,
    })

    # 2) hard_filter and soft_boost.
    hf_docids_all = []
    sb_docids_all = []

    for i, query in enumerate(queries_train):
        qid = query["id"]
        q_text = query["content"]

        # Features used to predict the query category.
        q_features = clf_vectorizer.transform([q_text])
        pred_cat = clf.predict(q_features)[0]
        cat_proba = clf.predict_proba(q_features)[0]

        base_ids = base_docids[i]
        base_sc  = base_scores[i]

        # hard_filter
        hf_ids, hf_sc = hard_filter(
            base_ids,
            base_sc,
            docs_by_id,
            predicted_category=pred_cat,
        )

        # soft_boost
        sb_ids, sb_sc = soft_boost(
            base_ids,
            base_sc,
            docs_by_id,
            category_proba=cat_proba,
            classes=clf.classes_,
        )

        hf_docids_all.append(hf_ids)
        sb_docids_all.append(sb_ids)

    metrics_hf = evaluate_with_classification(
        hf_docids_all,
        gt,
        query_ids_train,
        K,
        predicted_categories=predicted_cats,
        gt_categories=gt_categories,
    )

    metrics_sb = evaluate_with_classification(
        sb_docids_all,
        gt,
        query_ids_train,
        K,
        predicted_categories=predicted_cats,
        gt_categories=gt_categories,
    )

    results_rows.append({
        "method": method_name,
        "rerank": "hard_filter",
        **metrics_hf,
    })
    results_rows.append({
        "method": method_name,
        "rerank": "soft_boost",
        **metrics_sb,
    })

# Format results as a table.
import pandas as pd

df_results = pd.DataFrame(results_rows)
df_results = df_results.sort_values(["method", "rerank"]).reset_index(drop=True)
df_results = df_results[[
    "method",
    "rerank",
    "precision@k",
    "recall@k",
    "mrr@k",
    "query_category_accuracy",
]]
df_results[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]] = (
    df_results[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]].round(4)
)

df_results



In [ ]:
from src.retrieval.hybrid import fuse_rankings_rrf
from src.evaluation.evaluate import evaluate_run, indices_to_docids

weight_combos = [
    (3.5, 0.5),
    (3.0, 1.5),
    (3.0, 1.0),
    (3.5, 1.0),
    (3.3, 0.7), 
    (4.0, 1.0),
    (5.0, 0.5),
    (6.0, 0.5),
]

grid_rows = []
for we, wb in weight_combos:
    hybrid_idx, hybrid_sc = fuse_rankings_rrf(
        emb_indices=topk_indices_emb,
        bm25_indices=topk_indices_bm25p,
        k_out=100,
        rrf_k=20,
        weight_embeddings=we,
        weight_bm25=wb,
    )
    hybrid_docids = indices_to_docids(hybrid_idx, docs_subset)
    m = evaluate_run(hybrid_docids, gt, query_ids_train, K)
    score = 0.25 * (m["precision@k"] + m["recall@k"] + m["mrr@k"] + 0.9572)
    grid_rows.append({
        "we": we, "wb": wb,
        "precision": round(m["precision@k"], 4),
        "recall": round(m["recall@k"], 4),
        "mrr": round(m["mrr@k"], 4),
        "score_kaggle": round(score, 4),
    })

import pandas as pd
df_grid = pd.DataFrame(grid_rows).sort_values("score_kaggle", ascending=False)
print(df_grid.to_string(index=False))

This block inspects a single query to visually compare the top documents from the baseline vs. hard_filter vs. soft_boost reranking.


In [ ]:
# Inspect one query: baseline vs hard_filter vs soft_boost (Emb)

q_index = 0  # Index of the query to inspect.
query = queries_train[q_index]
qid = query["id"]

print(f"Query ID   : {qid}")
print(f"Query text : {query['content'][:300]}...\n")
print(f"Predicted category: {predicted_cats[qid]} (true: {query['category']})")

# Baseline Emb
base_ids    = pred_docids_bm25p[q_index]
base_scores = topk_scores_bm25p[q_index]

hf_ids, hf_scores = hard_filter(
    base_ids,
    base_scores,
    docs_by_id,
    predicted_category=predicted_cats[qid],
)

sb_ids, sb_scores = soft_boost(
    base_ids,
    base_scores,
    docs_by_id,
    category_proba=predicted_probas[qid],
    classes=clf.classes_,
)

def _describe_top(doc_ids, scores, k=10, title=""):
    print(f"\n=== {title} (top {k}) ===")
    for i, (doc_id, score) in enumerate(zip(doc_ids[:k], scores[:k]), start=1):
        d = docs_by_id[doc_id]
        print(f"{i:2d}. id={doc_id}  cat={d['category']}  score={score:.4f}")

_describe_top(base_ids, base_scores, k=10, title="Emb baseline")
_describe_top(hf_ids,   hf_scores,   k=10, title="Emb + hard_filter")
_describe_top(sb_ids,   sb_scores,   k=10, title="Emb + soft_boost")


## 4. Comparison with/without reranking

We now consolidate all results into a single table to compare retrieval methods 
(TF-IDF, BM25+, Embeddings, Hybrid RRF) across reranking strategies 
(no_rerank, hard_filter, soft_boost).

- Does reranking improve results?
- hard_filter vs soft_boost: which one works better? Why?
- Are there methods/configurations that clearly dominate?

Note: the cross-encoder reranking (`cross_encoder`) is evaluated separately in **Section 7**, as it requires significantly more compute time and uses a different interface.

In [ ]:
# Build final comparison DataFrame from results_rows computed in Section 3
df_results = pd.DataFrame(results_rows)
df_results = df_results.sort_values(["method", "rerank"]).reset_index(drop=True)
df_results = df_results[[
    "method",
    "rerank",
    "precision@k",
    "recall@k",
    "mrr@k",
    "query_category_accuracy",
]]
df_results[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]] = (
    df_results[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]].round(4)
)

df_results


## 5. Sensitivity analysis for soft_boost

In this final section, we study how the `soft_boost` reranking behaves when we change
the `boost_factor` hyperparameter. We fix the retrieval method (Embeddings and Hybrid RRF)
and evaluate several `boost_factor` values to see whether stronger boosting of the
predicted category can significantly improve precision@100, recall@100 or MRR@100.

In [ ]:
#Sensitivity analysis for soft_boost boost_factor (on embeddings + hybrid)

boost_factors = [-2, -1, -0.5, 0.5, 1.0, 1.5, 2.0]
methods_to_test = ["embeddings", "hybrid_rrf"]

softboost_rows = []

for method_name, base_docids, base_scores in [
    ("embeddings", pred_docids_emb,    topk_scores_emb),
    ("hybrid_rrf", pred_docids_hybrid, hybrid_scores),
]:
    if method_name not in methods_to_test:
        continue

    for bf in boost_factors:
        sb_docids_all = []

        for i, query in enumerate(queries_train):
            qid = query["id"]
            q_text = query["content"]

            q_features = clf_vectorizer.transform([q_text])
            cat_proba = clf.predict_proba(q_features)[0]

            base_ids = base_docids[i]
            base_sc  = base_scores[i]

            sb_ids, sb_sc = soft_boost(
                base_ids,
                base_sc,
                docs_by_id,
                category_proba=cat_proba,
                classes=clf.classes_,
                boost_factor=bf,
            )

            sb_docids_all.append(sb_ids)

        metrics_sb = evaluate_with_classification(
            sb_docids_all,
            gt,
            query_ids_train,
            K,
            predicted_categories=predicted_cats,
            gt_categories=gt_categories,
        )

        softboost_rows.append({
            "method": method_name,
            "boost_factor": bf,
            "precision@k": metrics_sb["precision@k"],
            "recall@k": metrics_sb["recall@k"],
            "mrr@k": metrics_sb["mrr@k"],
            "query_category_accuracy": metrics_sb["query_category_accuracy"],
        })

import pandas as pd

df_softboost = pd.DataFrame(softboost_rows)
df_softboost = df_softboost.sort_values(["method", "boost_factor"]).reset_index(drop=True)
df_softboost[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]] = (
    df_softboost[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]].round(4)
)

df_softboost

## 6. Additional experiments

This section explores three additional analyses to better understand the pipeline:
- **6.1** — Per-category retrieval metrics (does reranking help more for specific categories?)
- **6.2** — Effect of K (does reranking help more at smaller cutoffs?)
- **6.3** — Combined reranking: hard_filter + soft_boost (does chaining both strategies improve results?)

In [ ]:
# ===================== 6.1 Per-category metrics =====================
from src.classification.interfaces import CATEGORIES
from src.evaluation.evaluate import evaluate_run

per_cat_rows = []

for method_name, base_docids in [
    ("tfidf",      pred_docids_tfidf),
    ("bm25_plus",  pred_docids_bm25p),
    ("embeddings", pred_docids_emb),
    ("hybrid_rrf", pred_docids_hybrid),
]:
    for cat in CATEGORIES:
        cat_query_ids = [qid for qid in query_ids_train if gt_categories[qid] == cat]
        cat_indices   = [i for i, q in enumerate(queries_train) if q["id"] in cat_query_ids]
        cat_docids    = [base_docids[i] for i in cat_indices]

        if not cat_query_ids:
            continue

        metrics_cat = evaluate_run(cat_docids, gt, cat_query_ids, K)
        per_cat_rows.append({
            "method":       method_name,
            "category":     cat,
            "precision@k":  metrics_cat["precision@k"],
            "recall@k":     metrics_cat["recall@k"],
            "mrr@k":        metrics_cat["mrr@k"],
        })

df_per_cat = pd.DataFrame(per_cat_rows)
df_per_cat[["precision@k", "recall@k", "mrr@k"]] = (
    df_per_cat[["precision@k", "recall@k", "mrr@k"]].round(4)
)

print("Precision@100 per category and retrieval method:")
display(df_per_cat.pivot(index="category", columns="method", values="precision@k"))

print("\nRecall@100 per category and retrieval method:")
display(df_per_cat.pivot(index="category", columns="method", values="recall@k"))

print("\nMRR@100 per category and retrieval method:")
display(df_per_cat.pivot(index="category", columns="method", values="mrr@k"))

Overall, the per-category analysis confirms the global ranking of methods: embeddings and Hybrid RRF consistently outperform TF‑IDF and BM25+ in terms of precision, recall, and MRR across all categories. The effect of K is also monotonic: increasing K improves recall but slightly decreases precision, while MRR quickly stabilizes once K is large enough (around 50–100), meaning that most “useful” documents are already retrieved in the top ranks. These results suggest that our conclusions about the best retrieval configuration are robust across categories and cutoff values.

In [ ]:
# ===================== 6.2 Effect of K =====================
from src.evaluation.evaluate import evaluate_run

k_values = [5, 10, 20, 50, 100]

k_rows = []

for method_name, base_docids in [
    ("tfidf",      pred_docids_tfidf),
    ("bm25_plus",  pred_docids_bm25p),
    ("embeddings", pred_docids_emb),
    ("hybrid_rrf", pred_docids_hybrid),
]:
    for k_val in k_values:
        truncated_docids = [docids[:k_val] for docids in base_docids]
        metrics_k = evaluate_run(truncated_docids, gt, query_ids_train, k_val)
        k_rows.append({
            "method": method_name,
            "k":      k_val,
            "precision@k": metrics_k["precision@k"],
            "recall@k":    metrics_k["recall@k"],
            "mrr@k":       metrics_k["mrr@k"],
        })

df_k = pd.DataFrame(k_rows)
df_k[["precision@k", "recall@k", "mrr@k"]] = (
    df_k[["precision@k", "recall@k", "mrr@k"]].round(4)
)

df_k_pivot = df_k.pivot(index="k", columns="method", values="mrr@k")
print("MRR@k for different values of K:")
display(df_k_pivot)

The MRR@k curves show that our conclusions are stable across different cutoff values. 
For all k (5, 10, 20, 50, 100), Hybrid RRF and Embeddings consistently outperform BM25+ 
and TF-IDF, so the ranking of retrieval methods does not depend on k. MRR increases slightly 
when moving from k = 5 to k = 20–50, then quickly saturates, indicating that the first 
relevant document is almost always found within the top 20–50 results.

In [ ]:
# ===================== 6.3 Combined reranking: hard_filter → soft_boost =====================

combined_rows = []

for method_name, base_docids, base_scores in [
    ("tfidf",      pred_docids_tfidf,  topk_scores_tfidf),
    ("bm25_plus",  pred_docids_bm25p,  topk_scores_bm25p),
    ("embeddings", pred_docids_emb,    topk_scores_emb),
    ("hybrid_rrf", pred_docids_hybrid, hybrid_scores),
]:
    combined_docids_all = []

    for i, query in enumerate(queries_train):
        qid      = query["id"]
        q_text   = query["content"]

        q_features = clf_vectorizer.transform([q_text])
        pred_cat   = clf.predict(q_features)[0]
        cat_proba  = clf.predict_proba(q_features)[0]

        base_ids = base_docids[i]
        base_sc  = base_scores[i]

        # Step 1: hard_filter.
        hf_ids, hf_sc = hard_filter(
            base_ids,
            base_sc,
            docs_by_id,
            predicted_category=pred_cat,
        )

        # Step 2: soft_boost on the filtered results.
        sb_ids, sb_sc = soft_boost(
            hf_ids,
            hf_sc,
            docs_by_id,
            category_proba=cat_proba,
            classes=clf.classes_,
            boost_factor=1.5,
        )

        combined_docids_all.append(sb_ids)

    metrics_combined = evaluate_with_classification(
        combined_docids_all,
        gt,
        query_ids_train,
        K,
        predicted_categories=predicted_cats,
        gt_categories=gt_categories,
    )

    combined_rows.append({
        "method":  method_name,
        "rerank":  "hard_filter + soft_boost",
        "precision@k": metrics_combined["precision@k"],
        "recall@k":    metrics_combined["recall@k"],
        "mrr@k":       metrics_combined["mrr@k"],
        "query_category_accuracy": metrics_combined["query_category_accuracy"],
    })

df_combined = pd.DataFrame(combined_rows)
df_combined[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]] = (
    df_combined[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]].round(4)
)

print("Combined reranking (hard_filter → soft_boost):")
df_combined

The combined reranking strategy (hard_filter → soft_boost) does not bring additional gains 
compared to the individual strategies. For all retrieval methods, precision@100, recall@100 
and MRR@100 remain essentially identical to the `soft_boost` results, and the query category 
accuracy stays unchanged. This confirms that once documents are already filtered by category 
and the classifier is highly accurate, applying an extra soft_boost step has only a marginal 
effect on the final ranking.

## 7. Cross-Encoder Reranking

In this section, we apply a **cross-encoder** neural model to rerank the top-K candidates
retrieved by our best methods (Embeddings and Hybrid RRF).

Unlike `hard_filter` and `soft_boost` which use the classifier's category predictions,
the cross-encoder directly scores each (query, document) pair using a fine-tuned
transformer model (`cross-encoder/ms-marco-MiniLM-L-6-v2`), and reorders the candidates
by relevance score.

We test two configurations:

- `embeddings` + cross-encoder reranking (top-100 → rescore → top-100)
- `hybrid_rrf` + cross-encoder reranking (top-100 → rescore → top-100)

⚠️ This step is computationally expensive (~5–15 min on CPU for all train queries).
Reduce `k_in` or use a GPU for faster runs.

In [ ]:
# ===== Cross-encoder reranking on top of embeddings (experimental) =====
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

# 1) Load a small MS MARCO–style cross-encoder (binary relevance)
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(CE_MODEL)

K_in = 100   # how many candidates from embeddings to rerank
K_out = 100  # how many docs to keep after cross-encoder reranking

cross_docids_all = []

for i, query in enumerate(tqdm(queries_train, desc="Cross-encoder reranking")):
    q_text = query["content"]
    # take top-K_in candidates from embeddings
    cand_ids = pred_docids_emb[i][:K_in]

    # build (query, doc) pairs
    pairs = []
    for doc_id in cand_ids:
        doc = docs_by_id[doc_id]
        pairs.append((q_text, doc["content"]))

    # 2) Score pairs with the cross-encoder
    scores = cross_encoder.predict(pairs)

    # 3) Sort candidates by cross-encoder score
    sorted_idx = np.argsort(scores)[::-1]  # descending
    reranked_ids = [cand_ids[j] for j in sorted_idx[:K_out]]

    # pad to K if needed (optional, for consistency with evaluate_run)
    if len(reranked_ids) < K:
        # fill with remaining original embedding candidates
        remaining = [d for d in pred_docids_emb[i] if d not in reranked_ids]
        reranked_ids = reranked_ids + remaining[: K - len(reranked_ids)]

    cross_docids_all.append(reranked_ids)

# 4) Evaluate cross-encoder reranking
metrics_ce = evaluate_with_classification(cross_docids_all, gt, query_ids_train, K,predicted_categories=predicted_cats,
        gt_categories=gt_categories)
print("Cross-encoder reranking on top of embeddings:")
for k, v in metrics_ce.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
from src.classification.rerank import cross_encoder_rerank
import time

K_IN  = 100  # Number of candidates scored by the cross-encoder.
K_OUT = 100   # Number of results kept after reranking.

query_texts = [q["content"] for q in queries_train]

ce_rows = []

for method_name, base_docids in [
    ("embeddings", pred_docids_emb),
    ("hybrid_rrf", pred_docids_hybrid),
]:
    print(f"\n[CE] Reranking {method_name} ...")
    t0 = time.time()

    ce_docids, ce_scores = cross_encoder_rerank(
        query_texts=query_texts,
        topk_docids=base_docids,
        docs_by_id=docs_by_id,
        model_name=CE_MODEL,
        k_in=K_IN,
        k_out=K_OUT,
        batch_size=32,
    )
    elapsed = time.time() - t0
    print(f"[CE] Done in {elapsed:.1f}s")

    metrics_ce = evaluate_with_classification(
        ce_docids,
        gt,
        query_ids_train,
        K,
        predicted_categories=predicted_cats,
        gt_categories=gt_categories,
    )

    ce_rows.append({
        "method": method_name,
        "rerank": "cross_encoder",
        **metrics_ce,
    })
    print(f"  precision@{K} : {metrics_ce['precision@k']:.4f}")
    print(f"  recall@{K}    : {metrics_ce['recall@k']:.4f}")
    print(f"  mrr@{K}       : {metrics_ce['mrr@k']:.4f}")

print("\nCross-encoder reranking done.")

In [ ]:
# Merge results_rows (hard_filter / soft_boost) with ce_rows (cross_encoder).
df_ce = pd.DataFrame(ce_rows)
df_ce[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]] = (
    df_ce[["precision@k", "recall@k", "mrr@k", "query_category_accuracy"]].round(4)
)

# Global comparison table: embeddings + hybrid only, across all strategies.
df_all = pd.concat([df_results, df_ce], ignore_index=True)
df_all = df_all[df_all["method"].isin(["embeddings", "hybrid_rrf"])]
df_all = df_all.sort_values(["method", "rerank"]).reset_index(drop=True)

print("Full reranking comparison (embeddings & hybrid_rrf):")
display(
    df_all[[
        "method", "rerank",
        "precision@k", "recall@k", "mrr@k", "query_category_accuracy"
    ]].style.format(precision=4)
)

# Highlight best MRR per method
print("\nBest MRR per method:")
display(
    df_all.loc[df_all.groupby("method")["mrr@k"].idxmax(), 
               ["method", "rerank", "mrr@k", "recall@k"]].reset_index(drop=True)
)

### Observations

**1. Retrieval method ranking**  
Semantic methods clearly dominate lexical ones across all metrics:
- `hybrid_rrf` achieves the best MRR@100, followed by `embeddings`, 
  `bm25_plus` and `tfidf`.
- recall@100 follows the same order: hybrid_rrf = embeddings > bm25_plus > tfidf.
- Hybrid RRF matches embeddings on recall but improves MRR, confirming that BM25+ 
  lexical signals help rank the most relevant documents higher.

**2. Impact of reranking**  
- `recall@100` is **unchanged** across all reranking strategies for a given retrieval method. 
  This is expected: reranking only reorders the existing candidates, it cannot introduce new documents.
- `mrr@100` is slightly affected by reranking, but the changes are small (< 0.005 in most cases).
- `hard_filter` slightly **hurts** MRR for all methods: by filtering out documents from other 
  categories, it may remove some relevant documents that were ranked high.
- `soft_boost` is more conservative and performs slightly better than hard_filter in most cases, 
  but still does not outperform `no_rerank`.

**3. Why does reranking not help much here?**  
The classifier already achieves 95.7% query_category_accuracy on the training queries. 
This means that for most queries, the predicted category is already correct, so 
hard_filter and soft_boost are essentially a no-op: the top retrieved documents already 
belong to the right category. Reranking would be more impactful with a weaker classifier 
or on a more heterogeneous corpus.

Finally, Section 5 shows that varying the `soft_boost` `boost_factor` over a reasonable
range does not change the metrics significantly. This confirms that, in our setting,
classification-based reranking is not limited by the choice of hyperparameters, but
rather by the already high query category accuracy and the fact that reranking cannot
recover documents that were not retrieved in the top‑K in the first place.

**3. Conclusion** 

This section evaluated the full retrieval pipeline combining classification-based reranking 
strategies (hard_filter, soft_boost) on top of four retrieval methods (TF-IDF, BM25+, 
Embeddings, Hybrid RRF).

The experimental results consistently show that **Hybrid RRF remains the strongest 
configuration**, combining the semantic power of dense embeddings with the lexical 
precision of BM25+. Our final selected pipeline is:

- **Retrieval**: Hybrid RRF (Embeddings + BM25+)
- **Weights**: `weight_embeddings = 3.0`, `weight_bm25 = 1.5`
- **Reranking**: None (`no_rerank`)

This configuration achieves the best overall score across all four evaluation metrics 
(Precision@k, Recall@k, MRR@k, Query Category Accuracy).

Regarding classification-based reranking, results show that it does **not consistently 
improve retrieval quality** in our setup. The likely explanation is that the Hybrid RRF 
retriever already produces a near-optimal ranking for this domain-specific corpus, leaving 
little room for improvement through category-based filtering or score boosting.

> **Note**: `soft_boost` with `boost_factor = 0.5` achieves a score extremely close to 
> `no_rerank` (difference < 0.001), suggesting that a light soft boost neither significantly 
> helps nor hurts. The two configurations are practically equivalent, and either could be 
> used as the final pipeline.

This confirms the key finding of Phase 2: **improving the retrieval stage** (better 
bi-encoder, optimized fusion weights) yields far greater gains than adding a reranking 
layer on top of an already strong retriever.